# Lecture 13 Activity – Which model should decide the disputed essays?

QSS 20 · Modern Statistical Computing · Fall 2026

## The question

The lecture fitted three classifiers on five representations of the
Federalist essays. On the ten style words, all three found Madison's
known essays and agreed on ten of the 12 disputed essays. On the wide
TF-IDF matrices, logistic regression scored 1.0 on the training essays
and little above the baseline on the test essays. A report has to name
one model, one representation, and the evidence for both.

> **Which classifier, on which representation, should decide the 12
> disputed essays, and what in the held-out scores supports that choice?**

### How to work

Work in pairs for 30 minutes. Submit the best $k$, the three models'
training and test scores, the three classification reports, the 12
verdicts, and a short claim.
If you run out of time, finish Task 6 after class.

- The row unit is **one essay**. The setup reads the 85 files, labels the
  authors as the lecture did, builds the TF-IDF matrix with the stop list
  kept, and takes the ten style words as columns of it.
- Split the known essays once, with the lecture's settings:
  `test_size=0.25`, `random_state=42`, `stratify=`. Fit on the training
  rows; score on the test data; refit on all 65 only for the verdicts.
- In the starter cells, replace each `...` with code.

In [ ]:
import os
os.environ['OMP_NUM_THREADS'] = '1'

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import (
    GridSearchCV, cross_val_score, train_test_split)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier, plot_tree

RAW = Path('../../data/raw')

rows = []
for n in range(1, 86):
    path = RAW / 'federalist' / f'fp{n:02d}.txt'
    rows.append({'essay': n,
                 'text': path.read_text(encoding='utf-8')})
essays = pd.DataFrame(rows)

authors = {
    'Hamilton': ([1, 6, 7, 8, 9, 11, 12, 13, 15, 16, 17]
                 + list(range(21, 37)) + [59, 60, 61]
                 + list(range(65, 86))),
    'Madison': [10, 14] + list(range(37, 49)),
    'Jay': [2, 3, 4, 5, 64],
    'Joint': [18, 19, 20],
    'Disputed': list(range(49, 59)) + [62, 63],
}
label = {n: author for author, ns in authors.items() for n in ns}
essays['author'] = essays['essay'].map(label)
author = essays.set_index('essay')['author']

# The one matrix of the lecture: TF-IDF, stop list kept, as a DataFrame.
tfidf_vec = TfidfVectorizer(min_df=2)
tfidf = pd.DataFrame(tfidf_vec.fit_transform(essays['text']).toarray(),
                     columns=tfidf_vec.get_feature_names_out(),
                     index=essays['essay'])
style_words = ['upon', 'whilst', 'while', 'there', 'on', 'although',
               'always', 'commonly', 'consequently', 'enough']
style = tfidf[style_words]

known_ids = author.index[author.isin(['Hamilton', 'Madison'])]
disputed_ids = author.index[author == 'Disputed']
X = style.loc[known_ids]
y = author.loc[known_ids]
X_disputed = style.loc[disputed_ids]
classes = ['Hamilton', 'Madison']

# Check: 65 known rows, 12 disputed rows, ten columns.
X.shape, X_disputed.shape, y.value_counts().to_dict()

## Task 1 · The baseline and the split

Compute the baseline accuracy of a model that answers Hamilton for every
known essay. Then split `X` and `y` into training and test data with the
lecture's settings and print the author counts in each part.

How many Madison essays does the test set hold, and what would a model
that never answers Madison score on it?



<details><summary>Hint</summary>

`y.value_counts(normalize=True).max()`, then `train_test_split(X, y,
test_size=0.25, random_state=42, stratify=y)`. `y_train.value_counts()`
counts the authors.

</details>

In [ ]:
baseline = ...
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=..., random_state=..., stratify=...)
print(round(baseline, 3))
print(y_train.value_counts().to_dict(), y_test.value_counts().to_dict())

*Your answer:*

## Task 2 · The best $k$

Build a pipeline of `StandardScaler` and `KNeighborsClassifier`, and use
`GridSearchCV` with five folds on the training data to choose $k$ among
the odd numbers from 1 to 19. Report the best $k$, its cross-validated
accuracy, and the accuracy on the test data. Then print the
cross-validated accuracy for every $k$.

Where does the accuracy reach the baseline, and why does it fall there?



<details><summary>Hint</summary>

The parameter is named `kneighborsclassifier__n_neighbors` inside a
pipeline; `range(1, 20, 2)` gives the odd numbers.
`search.cv_results_['mean_test_score']` holds the ten scores.

</details>

In [ ]:
knn_search = GridSearchCV(
    make_pipeline(StandardScaler(), KNeighborsClassifier()),
    param_grid={'kneighborsclassifier__n_neighbors': range(...)},
    cv=5)
knn_search.fit(X_train, y_train)
print(knn_search.best_params_, round(knn_search.best_score_, 3))
print('test', round(knn_search.score(X_test, y_test), 3))

In [ ]:
pd.Series(knn_search.cv_results_['mean_test_score'],
          index=range(1, 20, 2)).round(3)

*Your answer:*

## Task 3 · Training accuracy against test accuracy

Build the lecture's logistic pipeline (`StandardScaler`, then
`LogisticRegression(max_iter=2000)`) and a
`DecisionTreeClassifier(max_depth=2, random_state=0)`, and fit both on
the training data. For kNN (the search of Task 2), the tree, and logistic
regression, print three numbers with `.score` and `cross_val_score`: the
accuracy on the training data, the five-fold mean on the training data,
and the accuracy on the test data.

Which of the three numbers would you report, and what does the gap
between the first and the other two say?



<details><summary>Hint</summary>

`model.score(X_train, y_train)`, `cross_val_score(model, X_train,
y_train, cv=5).mean()`, `model.score(X_test, y_test)`. For the search,
`knn_search.best_estimator_` is the fitted pipeline with the best $k$.

</details>

In [ ]:
logit = make_pipeline(StandardScaler(),
                      LogisticRegression(max_iter=2000))
logit.fit(X_train, y_train)
tree = DecisionTreeClassifier(max_depth=..., random_state=0)
tree.fit(X_train, y_train)
models = {'knn': knn_search.best_estimator_, 'tree': tree,
          'logistic': logit}
for name, model in models.items():
    folds = cross_val_score(model, ..., ..., cv=5)
    print(f'{name:9s} train {model.score(..., ...):.3f}'
          f'   five folds {folds.mean():.3f}'
          f'   test {model.score(..., ...):.3f}')

*Your answer:*

## Task 4 · Three classification reports

For kNN, the tree, and logistic regression, print the confusion matrix
on the test data with Hamilton first, then `classification_report`, and
list the essay numbers each model gets wrong.

Do the three models make the same mistakes, and which net is each?



<details><summary>Hint</summary>

`confusion_matrix(y_test, model.predict(X_test), labels=classes)`,
wrapped in a DataFrame with `classes` as index and columns;
`classification_report(y_test, model.predict(X_test))`.
`X_test.index[model.predict(X_test) != y_test]` lists the wrong essays.

</details>

In [ ]:
for name, model in models.items():
    out = model.predict(X_test)
    print(name, '  wrong:', list(X_test.index[out != y_test]))
    print(pd.DataFrame(confusion_matrix(y_test, out, labels=...),
                       index=classes, columns=classes))
    print(classification_report(..., ...))

*Your answer:*

## Task 5 · The twelve verdicts

Refit each model on all 65 known rows: kNN with the best $k$ from Task 2,
the tree with `max_depth=2`, and the logistic pipeline. Label the 12
disputed essays with each, add the logistic probability of Madison, and
count the essays on which the models disagree.

Which essays divide the models, and what in their style columns explains
the split?



<details><summary>Hint</summary>

Build each model again and call `.fit(X, y)`. Put the three
`.predict(X_disputed)` arrays in one DataFrame with `index=X_disputed.index`.
`verdicts[['knn', 'tree', 'logistic']].nunique(axis=1) > 1` marks the
disagreements. `X_disputed[['upon', 'there', 'on']]` shows the columns.

</details>

In [ ]:
best_k = knn_search.best_params_['kneighborsclassifier__n_neighbors']
knn_all = make_pipeline(StandardScaler(),
                        KNeighborsClassifier(n_neighbors=best_k))
tree_all = DecisionTreeClassifier(max_depth=2, random_state=0)
logit_all = make_pipeline(StandardScaler(),
                          LogisticRegression(max_iter=2000))
for model in [knn_all, tree_all, logit_all]:
    model.fit(..., ...)
verdicts = pd.DataFrame({
    'knn': knn_all.predict(X_disputed),
    'tree': tree_all.predict(X_disputed),
    'logistic': logit_all.predict(X_disputed),
    'p_madison': logit_all.predict_proba(X_disputed)[:, 1].round(2)},
    index=X_disputed.index)
split = verdicts[['knn', 'tree', 'logistic']].nunique(axis=1) > 1
print('essays with a disagreement:', list(verdicts.index[split]))
verdicts

In [ ]:
X_disputed.loc[split, ['upon', 'there', 'on']].round(4)

*Your answer:*

## Task 6 · Make a claim

Write four or five sentences. Name the model and the representation you
would report for the 12 disputed essays and the held-out evidence that
supports the choice. Say what the two disagreements mean for the report.
Then say why logistic regression on the 4,168-column matrix of the lecture
cannot settle the question even though it scored 1.0 on the training
essays.

*Your answer:*